# Task 3 — Executed Kafka evidence

This notebook reads the committed live-evidence summary. It does not call Docker or Kafka during the Jupyter Book build. The stored output was executed after the final LeRobot ingestion on 2026-07-25.

In [1]:
from pathlib import Path
import json
from IPython.display import display

def find_evidence():
    start = Path.cwd().resolve()
    for root in (start, *start.parents):
        candidate = root / 'docs' / 'evidence' / 'live_pipeline_summary.json'
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('docs/evidence/live_pipeline_summary.json not found')

evidence_path = find_evidence()
evidence = json.loads(evidence_path.read_text(encoding='utf-8'))
task = evidence['task3']
display({
    'captured_at': evidence['captured_at'],
    'repository_commit': evidence['provenance']['commit'],
    'topics': task['topics'],
    'initial_lerobot_publish': task['initial_lerobot_publish'],
    'connector': task['connector'],
})

{'captured_at': '2026-07-25T03:30:43.726347Z',
 'repository_commit': '0d383d09f2051444de211739196a28cc94736861',
 'topics': [{'name': 'cpg.nodes',
   'partitions': 3,
   'cleanup_policy': 'compact,delete',
   'retention_ms': 604800000,
   'end_offsets': {'0': 219695, '1': 217952, '2': 218999}},
  {'name': 'cpg.edges',
   'partitions': 3,
   'cleanup_policy': 'compact,delete',
   'retention_ms': 604800000,
   'end_offsets': {'0': 277086, '1': 277833, '2': 276927}},
  {'name': 'cpg.metadata',
   'partitions': 3,
   'cleanup_policy': 'compact,delete',
   'retention_ms': 2592000000,
   'end_offsets': {'0': 178, '1': 175, '2': 147}},
  {'name': 'cpg.errors',
   'partitions': 1,
   'cleanup_policy': 'delete',
   'retention_ms': 604800000,
   'end_offsets': {'0': 0}}],
 'initial_lerobot_publish': {'metadata_events': 490,
  'node_events': 655365,
  'edge_events': 830472,
  'parser_error_events': 0},
 'connector': {'name': 'cpg-neo4j-sink',
  'connector_state': 'RUNNING',
  'task_state': 'RUNNI

In [2]:
required_topics = {'cpg.nodes', 'cpg.edges', 'cpg.metadata', 'cpg.errors'}
observed_topics = {topic['name'] for topic in task['topics']}
assert observed_topics == required_topics
assert all(topic['partitions'] >= 1 for topic in task['topics'])
assert task['contract']['schema_version'] == '1.0'
assert task['contract']['utc_event_time'] is True
assert {'schema_version', 'event_time'} <= set(task['contract']['required_context'])
assert task['connector']['connector_state'] == 'RUNNING'
assert task['connector']['task_state'] == 'RUNNING'
assert all(lag == 0 for values in task['connector']['lag_by_topic'].values() for lag in values)
assert task['connector']['dlq_end_offset'] == 0
assert task['initial_lerobot_publish']['metadata_events'] == 490
assert task['initial_lerobot_publish']['parser_error_events'] == 0
result = {
    'status': 'PASS',
    'required_topics': sorted(observed_topics),
    'all_connector_lags_zero': True,
    'dlq_end_offset': 0,
}
display(result)

{'status': 'PASS',
 'required_topics': ['cpg.edges', 'cpg.errors', 'cpg.metadata', 'cpg.nodes'],
 'all_connector_lags_zero': True,
 'dlq_end_offset': 0}

## Reflection

Separate, versioned topics kept graph, metadata, parser failures and connector failures independently observable. Stable keys plus downstream upserts provide replay safety beyond producer idempotence alone.